# Prediksi Risiko Diabetes (Machine Learning)
Dataset: Pima Indians Diabetes. Model: Logistic Regression vs Random Forest.

## 1. Import & Load Data

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, joblib
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
cols = ["Pregnancies","Glucose","BloodPressure","SkinThickness","Insulin","BMI","DiabetesPedigreeFunction","Age","Outcome"]
df = pd.read_csv(url, names=cols)
df.head()

## 2. Eksplorasi Data

In [ ]:
print(df.shape)
print(df["Outcome"].value_counts())
df.describe()

In [ ]:
# Nilai 0 pada kolom ini tidak masuk akal secara medis -> dianggap data hilang
zero_cols = ["Glucose","BloodPressure","SkinThickness","Insulin","BMI"]
print((df[zero_cols]==0).sum())

plt.figure(figsize=(8,6))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Korelasi Antar Fitur")
plt.show()

## 3. Preprocessing
- 0 diganti NaN, lalu diisi median
- Fitur distandarisasi
- Split 80% train / 20% test

In [ ]:
df[zero_cols] = df[zero_cols].replace(0, np.nan)
X = df.drop("Outcome", axis=1)
y = df["Outcome"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)

## 4. Training Model

In [ ]:
models = {
    "Logistic Regression": Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler()), ("clf", LogisticRegression(max_iter=1000))]),
    "Random Forest": Pipeline([("imp", SimpleImputer(strategy="median")), ("clf", RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42))]),
}
results = {}
for name, m in models.items():
    m.fit(X_train, y_train)
    cv = cross_val_score(m, X_train, y_train, cv=5).mean()
    acc = accuracy_score(y_test, m.predict(X_test))
    results[name] = acc
    print(f"{name}: CV={cv:.3f} | Test Accuracy={acc:.3f}")

## 5. Evaluasi

In [ ]:
best_name = max(results, key=results.get)
best = models[best_name]
pred = best.predict(X_test)
print("Model terbaik:", best_name)
print(classification_report(y_test, pred, target_names=["Tidak Diabetes","Diabetes"]))

ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=["Tidak","Diabetes"]).plot(cmap="Blues")
plt.title(f"Confusion Matrix - {best_name}")
plt.show()

In [ ]:
# Fitur terpenting (Random Forest)
rf = models["Random Forest"].named_steps["clf"]
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values()
imp.plot(kind="barh", figsize=(7,4), title="Feature Importance (Random Forest)")
plt.show()

## 6. Simpan Model

In [ ]:
joblib.dump(best, "model_diabetes.joblib")
from google.colab import files
files.download("model_diabetes.joblib")